Remaking the MNIST Classifier made in FeedForwardMNIST but this time using Pytorch for experience with more relevent tools and comparing between the two

HOWEVER this time, we work directly with torch.tensor  objects with requires_grad=True
No using nn.Module or nn.Linear yet

In [26]:
import torch
from torchvision import datasets, transforms

In [27]:
W1 = torch.randn(784, 128) * 0.01
W1.requires_grad = True
W2 = torch.randn(128, 10) * 0.01
W2.requires_grad = True

b1 = torch.zeros(128)
b1.requires_grad = True
b2 = torch.zeros(10)
b2.requires_grad = True


Loading Data 

In [28]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)


In [29]:
X_train = train_dataset.data.float().view(-1, 784) / 255.0
y_train = train_dataset.targets

In [ ]:
epochs = 50 
batch_size = 32
lr = 0.05
for epoch in range(epochs):
    epoch_loss = 0
    for i in range (0, len(X_train), batch_size):
        # Slice up the data
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        # forward
        f1 = X_batch @ W1 + b1
        r1 = torch.clamp(f1, min=0)
        f2 = r1 @ W2 + b2
            # softmax
        exps = torch.exp(f2 - f2.max(dim=1, keepdim=True).values)
        probs = exps / exps.sum(dim=1, keepdim=True)
            # cross-entropy loss
        loss = -torch.log(probs[torch.arange(len(X_batch)), y_batch]).mean()

        epoch_loss += loss

        # backward
        loss.backward()

        # Update Weight
        with torch.no_grad():
            W1 -= lr * W1.grad
            W2 -= lr * W2.grad
            b1 -= lr * b1.grad
            b2 -= lr * b2.grad

        # Zero the gradients
        W1.grad.zero_()
        W2.grad.zero_()
        b1.grad.zero_()
        b2.grad.zero_()


    print(f"Loss[{epoch}]: {epoch_loss}")


Loss[0]: 1011.753173828125
Loss[1]: 458.74188232421875
Loss[2]: 346.75384521484375
Loss[3]: 276.1759338378906
Loss[4]: 228.6807098388672
